In [ ]:
# Run this once, then restart the kernel. Written for JupyterLab.
# anywidget is needed with Plotly 6 or newer.
%pip install ipywidgets anywidget

In [ ]:
# Interactive residuals panel with TWO-WAY linking (top <-> bottom)
## Drag a box in either chart: the same points are highlighted in the other one.
## Double-click a chart to clear.
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import statsmodels.api as sm
from sklearn.datasets import load_diabetes

# If you're in classic Jupyter or VS Code:
pio.renderers.default = "notebook_connected"  # try "vscode" or "jupyterlab" if needed

# --- Data & model
diab = load_diabetes()
df = pd.DataFrame(diab.data, columns=diab.feature_names)
df["target"] = diab.target

pred = "bmi"
X = sm.add_constant(df[[pred]])
y = df["target"]
m = sm.OLS(y, X).fit()
df["fitted"] = m.predict(X)
df["resid"]  = y - df["fitted"]

# --- Build FigureWidget with two linked charts
base = make_subplots(
    rows=2, cols=1, shared_xaxes=False,
    subplot_titles=(f"Actual vs Fitted (predictor: {pred})", "Residuals vs Fitted"),
    vertical_spacing=0.12
)
fig = go.FigureWidget(base)
fig.update_layout(height=650, width=800, dragmode="select", title="Linked Residuals Panel")

# Top: Actual vs Fitted (selectable)
top_scatter = go.Scatter(
    x=df["fitted"], y=df["target"],
    mode="markers", name="Actual vs Fitted",
    marker=dict(size=7, opacity=0.8),
    selected=dict(marker=dict(color="crimson", size=9)),
    unselected=dict(marker=dict(opacity=0.2)),
)

diag_line = go.Scatter(
    x=[df["fitted"].min(), df["fitted"].max()],
    y=[df["fitted"].min(), df["fitted"].max()],
    mode="lines", name="y = x", line=dict(dash="dash")
)

# Bottom: Residuals vs Fitted (selectable)
bottom_scatter = go.Scatter(
    x=df["fitted"], y=df["resid"],
    mode="markers", name="Residuals",
    marker=dict(size=7, opacity=0.8),
    selected=dict(marker=dict(color="crimson", size=9)),
    unselected=dict(marker=dict(opacity=0.2)),
)

fig.add_trace(top_scatter, 1, 1)
fig.add_trace(diag_line, 1, 1)
fig.add_trace(bottom_scatter, 2, 1)

fig.update_xaxes(title_text="Fitted (ŷ)", row=1, col=1)
fig.update_yaxes(title_text="Actual (y)", row=1, col=1)
fig.update_xaxes(title_text="Fitted (ŷ)", row=2, col=1)
fig.update_yaxes(title_text="Residual (y − ŷ)", row=2, col=1)
fig.add_hline(y=0, line_dash="dash", row=2, col=1)

# --- Highlight layers: one per chart, empty until points are selected in the OTHER chart.
# Plotly keeps control of `selectedpoints` on a chart that has a selection box, so the
# linked points are drawn as their own trace instead of by setting that property.
highlight = dict(
    mode="markers", marker=dict(color="crimson", size=9),
    showlegend=False, hoverinfo="skip",
    selected=dict(marker=dict(opacity=1)), unselected=dict(marker=dict(opacity=1)),
)
fig.add_trace(go.Scatter(x=[], y=[], name="linked", **highlight), 1, 1)
fig.add_trace(go.Scatter(x=[], y=[], name="linked", **highlight), 2, 1)

# --- Two-way linking handlers
top = fig.data[0]            # top_scatter
bottom = fig.data[2]         # bottom_scatter
top_linked = fig.data[3]     # highlight layer in the top chart
bottom_linked = fig.data[4]  # highlight layer in the bottom chart

# Plotly calls EVERY trace's handler on every selection, so each handler remembers
# what it saw last and only acts when the selection in its own chart has changed.
last = {"top": (), "bottom": ()}

def link_top_to_bottom(trace, points, selector):
    # Select in TOP -> highlight the same rows in BOTTOM
    inds = tuple(points.point_inds)
    if inds == last["top"]:
        return
    last["top"] = inds
    rows = df.iloc[list(inds)]
    with fig.batch_update():
        bottom_linked.x, bottom_linked.y = rows["fitted"], rows["resid"]
        top_linked.x, top_linked.y = [], []

def link_bottom_to_top(trace, points, selector):
    # Select in BOTTOM -> highlight the same rows in TOP
    inds = tuple(points.point_inds)
    if inds == last["bottom"]:
        return
    last["bottom"] = inds
    rows = df.iloc[list(inds)]
    with fig.batch_update():
        top_linked.x, top_linked.y = rows["fitted"], rows["target"]
        bottom_linked.x, bottom_linked.y = [], []

top.on_selection(link_top_to_bottom)
bottom.on_selection(link_bottom_to_top)

# Double-click a chart to clear its selection; clear the linked highlights with it
def clear_linked(trace, points):
    last["top"], last["bottom"] = (), ()
    with fig.batch_update():
        top_linked.x, top_linked.y = [], []
        bottom_linked.x, bottom_linked.y = [], []

top.on_deselect(clear_linked)
bottom.on_deselect(clear_linked)

fig  # In Jupyter, simply put `fig` as the last line to display
